The goal is to create a reliable **client-month layer** that later makes feature engineering easy. 

This notebook will produce the following tables:
- client_contracts_clean
- client_monthly_signals
- client_monthly_model_base unified table 

In [0]:
# Load raw tables 
from pyspark.sql import functions as F
from pyspark.sql.window import Window

clients = spark.table("clients")
contracts = spark.table("contracts")
monthly_engagement = spark.table("monthly_engagement")
support_tickets = spark.table("support_tickets")
nps_scores = spark.table("nps_scores")
renewals = spark.table("renewals")
client_health = spark.table("client_health")

In [0]:
# Inspect schemas 

clients.printSchema()
contracts.printSchema()
monthly_engagement.printSchema()
support_tickets.printSchema()
nps_scores.printSchema()
renewals.printSchema()
client_health.printSchema()

In [0]:
client_contracts_clean = (
    contracts
    .join(
        renewals.select(
            "client_id",
            "renewal_cycle_id",
            "contract_start_date",
            "contract_end_date",
            "annual_contract_value",
            "renewed_flag",
            "non_renewal_flag"
        ),
        on=["client_id", "contract_start_date", "contract_end_date", "annual_contract_value"],
        how="left"
    )
    .select(
        "client_id",
        "client_name",
        "industry",
        "region",
        "company_size",
        "eligible_employees",
        "contract_start_date",
        "contract_end_date",
        "contract_term_months",
        "annual_contract_value",
        "renewal_cycle_id",
        "renewed_flag",
        "non_renewal_flag"
    )
)

display(client_contracts_clean)

In [0]:
display(
    client_contracts_clean
    .groupBy("client_id")
    .count().alias("count")
    .orderBy(F.desc("count"))
)

In [0]:
client_monthly_signals = (
    monthly_engagement.alias("e")
    .join(
        support_tickets.alias("t"),
        on=["client_id", "month"],
        how="left"
    )
    .join(
        nps_scores.alias("n"),
        on=["client_id", "month"],
        how="left"
    )
    .select(
        "e.client_id",
        "e.month",
        "e.months_to_renewal",
        "e.latent_risk_segment",
        "e.enrolled_members",
        "e.active_members",
        "e.sessions",
        "e.webinar_attendance",
        "e.employer_comms_sent",
        "e.utilization_rate",
        "t.ticket_count",
        "t.high_severity_tickets",
        "n.nps"
    )
)

display(client_monthly_signals)

In [0]:
# from pyspark.sql.functions import col, sum, when

# def count_nulls(df):
#     null_counts = df.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns])
#     return null_counts

In [0]:
# Handle nulls intentionally 
# No tickets in a month should behave like 0 
client_monthly_signals = (
    client_monthly_signals
    .fillna({
        "ticket_count": 0,
        "high_severity_tickets": 0,
    }))

display(client_monthly_signals)

In [0]:
# Check NPS nulls
display(
    client_monthly_signals
    .filter(F.col("nps").isNull())
)

In [0]:
#Join monthly signals to contract context 
client_monthly_model_base = (
    client_monthly_signals.alias("m")
    .join(
        client_contracts_clean.alias("c"),
        on=["client_id"],
        how="left"
    )
    .filter(F.col("month") >= F.date_trunc("month", F.col("contract_start_date")))
    .filter(F.col("month") <= F.date_trunc("month", F.col("contract_end_date")))
    .select(
        "client_id"
        ,"client_name"
        ,"industry"
        ,"region"
        ,"company_size"
        ,"eligible_employees" 
        ,"annual_contract_value"
        ,"contract_start_date"
        ,"contract_end_date"
        ,"contract_term_months"
        ,"renewal_cycle_id"
        ,"renewed_flag"
        ,"non_renewal_flag"
        ,"month"
        ,"months_to_renewal"
        ,"latent_risk_segment"
        ,"enrolled_members"
        ,"active_members"
        ,"sessions"
        ,"webinar_attendance"
        ,"employer_comms_sent"
        ,"utilization_rate"
        ,"ticket_count" 
        ,"high_severity_tickets"
        ,"nps"

    )
)

display(client_monthly_model_base)

In [0]:
# Grain validation. Exactly one row per client per month. 
display(
    client_monthly_model_base
    .groupBy("client_id","month")
    .count()
    .filter(F.col("count") > 1))
    

In [0]:
print("clients:", clients.count())
print("contracts:", contracts.count())
print("monthly_engagement:", monthly_engagement.count())
print("client_monthly_signals:", client_monthly_signals.count())
print("client_monthly_model_base:", client_monthly_model_base.count())

In [0]:
display(
    client_monthly_model_base 
    .agg(
        F.min("month").alias("min_month"),
        F.max("month").alias("max_month")
    )
)

In [0]:
display(
    client_monthly_model_base.select(
        [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in client_monthly_model_base.columns]
    )
)

In [0]:
spark.sql("DROP TABLE IF EXISTS client_contracts_clean")
spark.sql("DROP TABLE IF EXISTS client_monthly_signals")
spark.sql("DROP TABLE IF EXISTS client_monthly_model_base")

In [0]:
client_contracts_clean.write.mode("overwrite").saveAsTable("client_contracts_clean")
client_monthly_signals.write.mode("overwrite").saveAsTable("client_monthly_signals")
client_monthly_model_base.write.mode("overwrite").saveAsTable("client_monthly_model_base")